## 深度研究

经典的跨业务 Agentic 用例之一！非常重要。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">商业应用</h2>
            <span style="color:#00bfff;">深度研究 Agent 广泛适用于任何业务领域，也适用于你的日常工作。你可以自己利用它！
            </span>
        </td>
    </tr>
</table>

In [29]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)

## OpenAI 托管工具

OpenAI Agents SDK 包含以下托管工具：

`WebSearchTool` 让 Agent 可以搜索网页。  
`FileSearchTool` 允许从你的 OpenAI Vector Stores 中检索信息。  
`ComputerTool` 允许自动化计算机操作任务，如截图和点击。

### 重要提示 - WebSearchTool 的 API 费用

OpenAI WebSearchTool 每次调用花费我 2.5 美分。接下来的 2 个实验可能会累计到 $2-$3。我们将在其他平台中使用免费和低成本的搜索工具，所以如果费用是个问题，你可以跳过运行这部分。另外，学生 Christian W. 指出 OpenAI 有时单次调用可能会收取多次搜索的费用，因此单次调用可能超过 2.5 美分。

费用信息：https://platform.openai.com/docs/pricing#web-search

In [ ]:
INSTRUCTIONS = "你是一名研究助理。给定一个搜索词，你需要搜索网络并生成一个简洁的结果摘要。摘要必须 2-3 段，少于 300 字。抓住要点。简洁书写，不需要完整的句子或完美的语法。这些内容将供撰写综合报告的人使用，因此你必须抓住核心内容，忽略任何冗余。除了摘要本身，不要包含任何额外的评论。"
# Agent = 大模型 + 工具 + 自动化编排
search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    # 强制模型必须调用工具，不能直接回答。
    # 默认情况下，模型可以自己决定是直接回复用户，
    # 还是调用工具。设置 tool_choice="required" 最少调用一个工具，否则模型会被惩罚。
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

### 和往常一样，查看追踪记录

https://platform.openai.com/traces

### 现在我们将使用结构化输出，并包含字段的描述说明

In [ ]:
# 请参阅上面关于 WebSearchTool 费用的说明

HOW_MANY_SEARCHES = 3 

INSTRUCTIONS = f"你是一名乐于助人的研究助理。给定一个查询，想出一组最能回答该查询的网络搜索词。输出 {HOW_MANY_SEARCHES} 个搜索词。"

# 使用 Pydantic 来定义响应的 Schema —— 这被称为"结构化输出"
# 
class WebSearchItem(BaseModel):
    # 为什么搜索
    reason: str = Field(description="为什么这个搜索对查询很重要的理由。")
    # 实际搜索
    query: str = Field(description="用于网络搜索的搜索词。")


class WebSearchPlan(BaseModel):
    # 告诉大模型生成结构化输出搜索词的要求(原因)，
    # 大模型会将搜索词拆分成HOW_MANY_SEARCHES份
    # 每份的结构和WebSearchItem一样，包含搜索词和原因
    # ❤ 如果要进行多次搜索，给出每次搜索的原因和搜索词，
    # 可以让模型更有针对性地进行搜索，从而提高搜索结果的相关性和质量。
    # searche如果是一次搜索也要交代
    # 为了最佳回答查询(目标) 而要执行的网络搜索列表(告诉模型是列表)。
    searches: list[WebSearchItem] = Field(description="为了最佳回答查询而要执行的网络搜索列表。")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [ ]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)
# 这是一个搜索计划返回定义好的结构格式
"""
│  searches=[                                           │
│    WebSearchItem(reason='...', query='...'),          │
│    WebSearchItem(reason='...', query='...'),          │
│    WebSearchItem(reason='...', query='...'),          │
│  ]   
"""

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ 发送给定主题和 HTML 内容的邮件 """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("ed@edwarddonner.com") # 改为你已验证的发送者邮箱
    to_email = To("ed.donner@gmail.com") # 改为你的收件人邮箱
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [ ]:
send_email

In [ ]:
INSTRUCTIONS = """你可以基于详细报告发送一封格式良好的 HTML 邮件。
你会收到一份详细报告。你需要使用工具发送一封邮件，将报告转换成干净、排版良好的 HTML 格式，并配上合适的主题行。"""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)


In [ ]:
INSTRUCTIONS = (
    "你是一名资深研究员，负责为研究查询撰写连贯的报告。"
    "你将收到原始查询以及研究助理完成的初步研究。\n"
    "你应首先为报告制定大纲，描述报告的结构和流程。"
    "然后生成报告并将其作为最终输出返回。\n"
    "最终输出应为 markdown 格式，内容应详实深入。"
    "目标是 5-10 页的内容，至少 1000 字。"
)

# 还是结构化输出，告诉模型要怎么写报告，写什么内容，怎么写
class ReportData(BaseModel):
    short_summary: str = Field(description="研究发现的简短 2-3 句摘要。")

    markdown_report: str = Field(description="最终报告")

    follow_up_questions: list[str] = Field(description="建议进一步研究的主题")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### 接下来的 3 个函数将使用 planner_agent 和 search_agent 来规划和执行搜索

In [ ]:
async def plan_searches(query: str):
    """ 使用 planner_agent 规划需要为查询执行哪些搜索 """
    print("正在规划搜索...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"将执行 {len(result.final_output.searches)} 次搜索")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ 对搜索计划中的每个项调用 search() """
    print("正在搜索...")
    # Python 函数的执行时机和定义时机是两回事，
    # 只要后面有定义 search() 函数，这里就可以先调用它
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("搜索完成")
    return results

async def search(item: WebSearchItem):
    """ 使用搜索 Agent 对搜索计划中的每个项执行网络搜索 """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### 接下来的 2 个函数负责撰写报告并发送邮件

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ 使用写作 Agent 根据搜索结果撰写报告"""
    print("正在构思报告...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("报告撰写完成")
    return result.final_output

async def send_email(report: ReportData):
    """ 使用邮件 Agent 发送包含报告的邮件 """
    print("正在编写邮件...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("邮件已发送")
    return report

### 好戏开场！

In [ ]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    # 返回结构化搜索计划
    search_plan = await plan_searches(query)
    # 返回搜索后得到的结果列表
    search_results = await perform_searches(search_plan)
    # 遍历搜索结果，撰写报告
    report = await write_report(query, search_results)
    # 发送
    await send_email(report)  
    print("Hooray!")




### 和往常一样，查看追踪记录

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">恭喜你取得进步，还有一个请求</h2>
            <span style="color:#00cc00;">你已经到达了课程的一个重要时刻；你使用最新的 Agent 框架创建了一个有价值的 Agent。你已经提升了技能，解锁了新的商业可能性。花点时间庆祝你的成功！<br/><br/>有件事我要问你——如果我不提这个，我的编辑会打我的。如果你能在 Udemy 上给课程评分，我将万分感激：这是 Udemy 决定是否向其他人展示课程的最重要方式，影响巨大。<br/><br/>另外，如果你愿意，记得<a href="https://www.linkedin.com/in/eddonner/">在 LinkedIn 上联系我</a>！如果你想发布关于课程进展的内容，请标记我，我会参与互动来增加你的曝光度。
            </span>
        </td>
    </tr></table>